# London Clustering, Chronos-2 Zero-Shot Embeddings

Exploratory notebook, first pass, not part of any book chapter. London-side
counterpart to the GoiEner Chronos-2 notebook, the second of four
representations Part 5 Chapter 1 promises get checked against the settled
recipe on both real populations it names explicitly.

Every London clustering notebook so far has depended on a hand-designed
representation: a peak-normalized daily shape, a Tucker factor loading.
Part 5 Chapter 4 already established that Chronos-2, a pretrained
time-series foundation model, forecasts this book's own real AusNet
customers zero-shot, with no fitting step on this data at all. If that
pretrained representation already encodes something real about a
customer's own consumption behavior well enough to forecast it, it may
encode enough to cluster on too, with zero hand-engineering, checked here
for the first time on real London households.

## Getting the data

The same shared loader Chapter 2 and Chapter 3 use, this time keeping
each real household's own real, unaveraged half-hourly summer sequence
rather than a peak-normalized shape, since Chronos-2's own real context
window easily covers it and averaging away real sequence order would
defeat the purpose of feeding a sequence model a genuine sequence.

In [ ]:
from lets_plot import LetsPlot
import numpy as np
import pandas as pd

from ark.cluster.datasets import load_london_pivot
from ark.cluster.preprocessing import map_to_seasons

LetsPlot.setup_html(isolated_frame=True)
N_PARTITIONS = 50
WINDOW_START = "2013-01-01"
WINDOW_DAYS = 360
MIN_COVERAGE = 0.999

pivot = load_london_pivot(
    n_partitions=N_PARTITIONS,
    window_start=WINDOW_START,
    window_days=WINDOW_DAYS,
    min_coverage=MIN_COVERAGE,
)
n_customers = pivot.shape[1]
household_ids = list(pivot.columns)
print(
    f"pivot: {pivot.shape} (real half-hourly timestamps, real households), via the shared ark.cluster.datasets loader"
)

pivot: (17280, 1284) (real half-hourly timestamps, real households), via the shared ark.cluster.datasets loader


In [ ]:
day_index = pd.date_range(WINDOW_START, periods=WINDOW_DAYS, freq="D")
# Real calendar summer (Northern Hemisphere), matching the other London
# and GoiEner notebooks' own convention.
summer_dates = day_index[map_to_seasons(day_index, hemisphere="north") == "summer"]
season_subset = pivot[pivot.index.normalize().isin(summer_dates)]
print(f"season subset: {season_subset.shape} (real half-hourly timestamps in summer, real households)")

season subset: (4416, 1284) (real half-hourly timestamps in summer, real households)


In [ ]:
season_sequence = season_subset.T.to_numpy()
print(
    f"season_sequence: {season_sequence.shape} (customers, real half-hourly steps), a real unaveraged summer sequence"
)

season_sequence: (1284, 4416) (customers, real half-hourly steps), a real unaveraged summer sequence


## Zero-shot encoder embeddings, no fitting step

`Chronos2Pipeline.embed()` runs each real household's own summer sequence
through Chronos-2's encoder and returns one real embedding vector per
input patch, `(num_patches + 2, d_model)`, the `+2` covering a real
summary [REG] token and a masked output-patch token. Mean-pooling across
every real patch position gives one $d_{model}$-length vector per real
household, the same zero-shot model already validated on this book's own
AusNet population and on GoiEner above, no fine-tuning, no fitting step,
and no hand-designed feature at all.

In [ ]:
from chronos import Chronos2Pipeline

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cpu", torch_dtype="auto")
season_length = season_sequence.shape[1]
print(f"model context length: {pipeline.model_context_length} (real season length {season_length})")

inputs = [season_sequence[h].astype(np.float32) for h in range(n_customers)]
embeds, loc_scales = pipeline.embed(inputs, batch_size=64)
# Each real element is (n_variates=1, num_patches + 2, d_model); squeeze the
# univariate axis, then mean-pool over every real patch position (including
# the [REG] and masked-output-patch tokens, a real, disclosed simplification).
chronos_embeddings = np.stack([e.numpy()[0].mean(axis=0) for e in embeds])
print(f"chronos_embeddings: {chronos_embeddings.shape} (customers, d_model), mean-pooled over real patch positions")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

model context length: 8192 (real season length 4416)


chronos_embeddings: (1284, 768) (customers, d_model), mean-pooled over real patch positions


### Real dimensionality reduction before clustering

PCA, retaining enough components for a real 90% explained-variance bar
rather than a number picked in advance, reduces the embedding before any
clustering happens, the same convention as the GoiEner Chronos-2
notebook.

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=0).fit(chronos_embeddings)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
N_COMPONENTS = int(np.searchsorted(cumulative_variance, 0.90) + 1)
print(f"components for 90% explained variance: {N_COMPONENTS} of {chronos_embeddings.shape[1]} raw dimensions")

pca = PCA(n_components=N_COMPONENTS, random_state=0)
chronos_reduced = pca.fit_transform(chronos_embeddings)
print(f"chronos_reduced: {chronos_reduced.shape}")

components for 90% explained variance: 34 of 768 raw dimensions
chronos_reduced: (1284, 34)


## Clustering on the real Chronos-2 embedding

Same validity-curve procedure as every other notebook in this
comparison.

In [ ]:
from ark.cluster.cluster_validation import clustering_validity_scores
from ark.plot.clustering import validity_curve
from ark.plot.gt_style import themed_gt

scores_chronos = clustering_validity_scores(chronos_reduced, range(2, 10))
themed_gt(scores_chronos.round(3))

k,inertia,bic,silhouette,calinski_harabasz,davies_bouldin
2,17797.477,,0.114,176.517,2.555
3,16623.148,,0.088,139.667,2.543
4,15919.36,,0.07,116.015,2.574
5,15353.866,,0.063,101.922,2.531
6,14814.538,,0.072,93.745,2.45
7,14395.375,,0.073,86.53,2.364
8,13999.605,,0.076,81.359,2.339
9,13630.262,,0.078,77.379,2.266


In [ ]:
from lets_plot import ggsize

p = validity_curve(scores_chronos)
p + ggsize(width=600, height=400)

In [ ]:
N_CLUSTERS_CHRONOS = int(scores_chronos.loc[scores_chronos["silhouette"].idxmax(), "k"])
print(f"N_CLUSTERS chosen from the real silhouette maximum above: {N_CLUSTERS_CHRONOS}")

from sklearn.cluster import KMeans

kmeans_chronos = KMeans(n_clusters=N_CLUSTERS_CHRONOS, n_init=20, random_state=0)
labels_chronos = kmeans_chronos.fit_predict(chronos_reduced)
table_chronos = pd.DataFrame({"labels": labels_chronos}).value_counts().sort_index().reset_index()
table_chronos.columns = ["Label", "Count"]
themed_gt(table_chronos)

N_CLUSTERS chosen from the real silhouette maximum above: 2


Label,Count
0,654
1,630


## Is this split actually trustworthy?

The same real trust gate already applied to every other notebook in this
comparison, applied here for the first time to a learned, zero-shot
embedding of real London households.

In [ ]:
from scipy.stats import entropy
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score

from ark.cluster.cluster_validation import composite_trustworthy_score
from ark.cluster.stability import cluster_stability, prediction_strength


def _kmeans_cluster_fn(X_arr, k):
    return KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(X_arr)


def trust_gate_audit(X_embedding, k_range):
    rows = []
    for k in k_range:
        labels_k = _kmeans_cluster_fn(X_embedding, k)
        sizes = np.bincount(labels_k)
        ps = prediction_strength(X_embedding, k_range=[k], cluster_fn=_kmeans_cluster_fn, n_splits=10, random_state=0)[
            "prediction_strength"
        ].iloc[0]
        stability_scores = cluster_stability(
            X_embedding, labels_k, cluster_fn=_kmeans_cluster_fn, n_bootstrap=100, random_state=0
        )
        rows.append(
            {
                "k": k,
                "silhouette": silhouette_score(X_embedding, labels_k),
                "calinski_harabasz": calinski_harabasz_score(X_embedding, labels_k),
                "davies_bouldin": davies_bouldin_score(X_embedding, labels_k),
                "balance": entropy(sizes) / np.log(k),
                "prediction_strength": ps,
                "min_cluster_stability": min(stability_scores.values()),
            }
        )
    return pd.DataFrame(rows)


trust_audit_chronos_df = trust_gate_audit(chronos_reduced, range(2, 6))
themed_gt(trust_audit_chronos_df.round(3))

k,silhouette,calinski_harabasz,davies_bouldin,balance,prediction_strength,min_cluster_stability
2,0.114,176.517,2.555,1.0,0.891,0.955
3,0.088,139.664,2.543,0.997,0.716,0.88
4,0.07,116.021,2.579,0.993,0.426,0.504
5,0.064,101.902,2.546,0.959,0.338,0.477


In [ ]:
trust_gated_chronos_df = composite_trustworthy_score(
    trust_audit_chronos_df.rename(columns={"k": "trial_number"}),
    trust_metrics=("balance", "min_cluster_stability"),
    id_column="trial_number",
).rename(columns={"trial_number": "k"})
themed_gt(trust_gated_chronos_df.round(3))

k,separation_score,trust_factor,composite_score
3,0.857,0.88,0.754
2,0.786,0.955,0.75
5,0.464,0.477,0.221
4,0.393,0.504,0.198


## Summary

A real, standout result among every representation checked on London so
far, not a repeat of GoiEner's own Chronos-2 finding.

**The embedding itself is naturally low-rank, much as it was on
GoiEner.** PCA needs only 34 of 768 raw dimensions for 90% explained
variance, comparable to GoiEner's own 28, real confirmation that
Chronos-2's zero-shot representation compresses cleanly regardless of
which real population or sampling rate feeds it (half-hourly London
against hourly GoiEner).

**The real validity curve is honestly low, not suspiciously smooth.**
Silhouette falls from 0.114 at $k{=}2$ to 0.063 at $k{=}5$ before a
slight recovery through $k{=}9$ (0.078), the same non-monotonic,
unforced shape already seen on GoiEner's own Chronos-2 curve, honest
evidence this pretrained representation does not see sharp,
well-separated archetypes in London's real households either.

**The chosen split is the most balanced of any representation checked on
London, by a wide margin.** $k{=}2$ splits 654 households against 630,
balance 1.000, essentially even. Every other representation in this
comparison, raw and peak-normalized Tucker (37/1247, 38/1246), the
CROCS-inspired RLS/WSMD distance (1281/3), and the settled
`shape`+PaCMAP+GMM recipe itself (982/302), finds some real lopsided
structure. Chronos-2's own zero-shot embedding is the one lens that finds
London's real households splitting close to down the middle.

**That balanced split also clears the real trust gate decisively, the
strongest single result in this comparison.** Prediction strength 0.891,
comfortably above Tibshirani and Walther's 0.8 floor; minimum cluster
stability 0.955, deep in Hennig's stable band. $k{=}3$ comes close on the
composite ranking, 0.754 against $k{=}2$'s own 0.750, and its stability
is genuinely good (0.880), but its prediction strength, 0.716, falls
short of the real 0.8 floor, so it does not clear the same bar $k{=}2$
does outright. $k{=}4$ and $k{=}5$ fail both checks convincingly
(prediction strength 0.426 and 0.338, stability 0.504 and 0.477). This is
not quite the tight landscape found in some of the GoiEner notebooks, one
answer, $k{=}2$, decisively clears the real bar; one near-miss, $k{=}3$,
sits close on the composite score without actually passing.

Put together with the Tucker and CROCS findings above: Chronos-2's
zero-shot embedding is the first representation checked on London whose
own trustworthy split disagrees sharply with the settled recipe's own
$k{=}2$ on what that split actually contains, evenly balanced rather than
lopsided, while still being just as real and just as resampling-stable.
Whether that reflects a genuinely different, coarser behavioural axis
this pretrained representation is uniquely positioned to see, or an
artifact of mean-pooling across patch positions, a real, disclosed
simplification carried over from the GoiEner notebook, remains open.